# Dataset Generation – Industrial Chiller Digital Twin
Autor: Vitor Mazon

Este notebook gera um dataset sintético representando medições
operacionais de um sistema de refrigeração industrial.

Sistemas de refrigeração industrial geram grande volume de dados operacionais.
Esses dados podem ser utilizados para análise de eficiência energética
e detecção de anomalias.

Neste projeto será desenvolvido um Digital Twin simplificado de um chiller
industrial. O primeiro passo consiste em gerar um dataset sintético
com comportamento físico plausível.

## Variáveis simuladas

| Variável | Descrição |
|--------|-----------|
| timestamp | instante da medição |
| temp_entrada_c | temperatura de entrada da água |
| temp_saida_c | temperatura de saída da água |
| vazao_m3_h | vazão volumétrica |
| potencia_kw | potência elétrica do sistema |
| pressao_oleo_bar | pressão do óleo do compressor |
| nivel_tanque_pct | nível do tanque |

## Criação da estrutura do Lakehouse

Nesta etapa criamos o catalog, schema e volume
onde os dados do projeto serão armazenados.

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS analytics;

CREATE SCHEMA IF NOT EXISTS analytics.digital_twin;

CREATE VOLUME IF NOT EXISTS analytics.digital_twin.data;

## Geração da estrutura

In [0]:
import os

base_path = "/Volumes/analytics/digital_twin/data"

os.makedirs(f"{base_path}/raw", exist_ok=True)
os.makedirs(f"{base_path}/silver", exist_ok=True)
os.makedirs(f"{base_path}/gold", exist_ok=True)

print("Estrutura de diretórios criada.")

## Geração do dataset

In [0]:
import sys
sys.path.append("../src")

from simularDados import gerar_dados_fake

arquivo = gerar_dados_fake(
    output_dir="/Volumes/analytics/digital_twin/data/raw"
)

print("Arquivo gerado em:", arquivo)

## Inspeção inicial

In [0]:
import pandas as pd

df = pd.read_csv(arquivo)

display(df.head())

In [0]:
df.describe()

In [0]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

df["timestamp"] = pd.to_datetime(df["timestamp"])

fig, ax = plt.subplots(figsize=(14, 4))

ax.plot(df["timestamp"], df["vazao_m3_h"], linewidth=1.0)

ax.set_title("Vazão de Água ao Longo do Tempo")
ax.set_xlabel("Data")
ax.set_ylabel("Vazão (m³/h)")

ax.xaxis.set_major_locator(mdates.DayLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%d/%m"))

ax.grid(True, linestyle="--", alpha=0.4)
ax.set_ylim(48, 107)

plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
plt.tight_layout()
plt.show()

O dataset gerado apresenta comportamento coerente
com um sistema de refrigeração industrial e inclui
eventos operacionais que poderão ser utilizados
nas etapas seguintes de análise e detecção de anomalias.